## Research questions

1. Which numerical audio features have the strongest relationships with track popularity?
2. Do explicit tracks have a different average popularity than non-explicit tracks?
3. How accurately can popularity be predicted from audio features using multiple linear regression?

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
RANDOM_STATE = 3022

## 1. Load and inspect the data

Place `dataset.csv` in the same folder as this notebook before running it.

In [ ]:
DATA_PATH = Path("dataset.csv")
raw = pd.read_csv(DATA_PATH)
print(f"Raw shape: {raw.shape[0]:,} rows × {raw.shape[1]} columns")
raw.head()

In [ ]:
raw.info()
raw.isna().sum()[raw.isna().sum() > 0]

## 2. Data cleaning

Rows missing identifying information are removed. Because repeated Spotify track IDs represent the same track, only the first ID is kept so repeated tracks do not receive extra weight. Duration is converted from milliseconds to minutes.

In [ ]:
duplicate_ids = raw.duplicated("track_id").sum()

df = (raw
      .drop(columns=["Unnamed: 0"])
      .dropna(subset=["track_id", "artists", "album_name", "track_name"])
      .drop_duplicates(subset="track_id", keep="first")
      .copy())
df["duration_min"] = df["duration_ms"] / 60000

print(f"Repeated track IDs removed: {duplicate_ids:,}")
print(f"Cleaned shape: {len(df):,} unique tracks × {df.shape[1]} columns")
print(f"Genres represented after cleaning: {df['track_genre'].nunique()}")

## 3. Exploratory analysis

In [ ]:
features = ['duration_min', 'danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'mode', 'time_signature']
df[["popularity"] + features].describe().T.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(df["popularity"], bins=25, color="#1DB954", edgecolor="white", ax=ax)
ax.axvline(df["popularity"].mean(), color="#191414", linestyle="--",
           label=f"Mean = {df['popularity'].mean():.1f}")
ax.set(title="Distribution of Spotify Track Popularity",
       xlabel="Popularity score (0–100)", ylabel="Number of unique tracks")
ax.legend();
plt.show()

Popularity is spread widely across its 0–100 scale. The mean and median are both near 33

## 4. Research Question 1: Feature relationships with popularity

In [ ]:
correlations = (df[["popularity"] + features]
                .corr()["popularity"]
                .drop("popularity")
                .sort_values(key=lambda s: s.abs(), ascending=False))
correlations.round(4)

In [ ]:
plot_corr = correlations.sort_values()
fig, ax = plt.subplots(figsize=(8, 5.5))
colors = ["#d95f5f" if value < 0 else "#1DB954" for value in plot_corr]
ax.barh(plot_corr.index, plot_corr.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set(title="Correlation of Audio Features with Popularity",
       xlabel="Pearson correlation", ylabel="", xlim=(-0.25, 0.25));
plt.show()

All correlations are weak. 

## 5. Research Question 2: Explicit versus non-explicit tracks

In [ ]:
explicit = df.loc[df["explicit"], "popularity"].astype(float)
nonexplicit = df.loc[~df["explicit"], "popularity"].astype(float)

comparison = pd.DataFrame({
    "group": ["Explicit", "Non-explicit"],
    "n": [len(explicit), len(nonexplicit)],
    "mean": [explicit.mean(), nonexplicit.mean()],
    "standard deviation": [explicit.std(), nonexplicit.std()]
})
comparison.round(3)

In [ ]:
test = stats.ttest_ind(explicit, nonexplicit, equal_var=False)
mean_difference = explicit.mean() - nonexplicit.mean()
se = np.sqrt(explicit.var(ddof=1)/len(explicit) + nonexplicit.var(ddof=1)/len(nonexplicit))
df_welch = (explicit.var(ddof=1)/len(explicit) + nonexplicit.var(ddof=1)/len(nonexplicit))**2 / (
    (explicit.var(ddof=1)/len(explicit))**2/(len(explicit)-1) +
    (nonexplicit.var(ddof=1)/len(nonexplicit))**2/(len(nonexplicit)-1))
critical = stats.t.ppf(0.975, df_welch)
ci = (mean_difference - critical*se, mean_difference + critical*se)
pooled_sd = np.sqrt(((len(explicit)-1)*explicit.var(ddof=1) +
                     (len(nonexplicit)-1)*nonexplicit.var(ddof=1)) /
                    (len(explicit)+len(nonexplicit)-2))
cohen_d = mean_difference / pooled_sd

print(f"Mean difference: {mean_difference:.3f}")
print(f"95% CI: ({ci[0]:.3f}, {ci[1]:.3f})")
print(f"Welch t({df_welch:.1f}) = {test.statistic:.3f}")
print(f"p-value = {test.pvalue:.3e}")
print(f"Cohen's d = {cohen_d:.3f}")

In [ ]:
plot_df = df.assign(Lyrics=np.where(df["explicit"], "Explicit", "Non-explicit"))
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=plot_df, x="Lyrics", y="popularity", hue="Lyrics",
            palette=["#777777", "#1DB954"], legend=False,
            showfliers=False, ax=ax)
ax.set(title="Popularity by Explicit-Lyrics Status", xlabel="", ylabel="Popularity score");
plt.show()

## 6. Research Question 3: Multiple linear regression

In [ ]:
X = df[features]
y = df["popularity"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("regression", LinearRegression())
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
baseline = np.repeat(y_train.mean(), len(y_test))

metrics = pd.DataFrame({
    "model": ["Mean-only baseline", "Multiple linear regression"],
    "RMSE": [mean_squared_error(y_test, baseline)**0.5,
             mean_squared_error(y_test, pred)**0.5],
    "MAE": [mean_absolute_error(y_test, baseline),
            mean_absolute_error(y_test, pred)],
    "R-squared": [r2_score(y_test, baseline), r2_score(y_test, pred)]
})
metrics.round(3)

In [ ]:
coefficients = pd.Series(model.named_steps["regression"].coef_,
                         index=features,
                         name="standardized coefficient")
coefficients.sort_values(key=lambda s: s.abs(), ascending=False).round(3)

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
idx = rng.choice(len(y_test), size=min(6000, len(y_test)), replace=False)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test.iloc[idx], pred[idx], alpha=0.18, s=14,
           color="#1DB954", edgecolors="none")
ax.plot([0, 100], [0, 100], "--", color="#191414", label="Perfect prediction")
ax.set(title=f"Actual vs. Predicted Popularity (Test $R^2$ = {r2_score(y_test, pred):.3f})",
       xlabel="Actual popularity", ylabel="Predicted popularity",
       xlim=(0, 100), ylim=(0, 100))
ax.legend();
plt.show()

## 7. Conclusions and limitations

The strongest individual association is the weak negative relationship between instrumentalness and popularity. Explicit tracks are slightly more popular on average, but its marginal. Multiple linear regression performs only slightly better than predicting the training-set mean for every track.

limitations include nonrandom dataset construction, changing popularity scores, the removal of repeated tracks, omitted variables such as artist fame, release date, marketing, playlist placement, and listener behavior. Future work could add release and artist-level information
